# NRC-VAD Arousal Intensity

This notebook estimates annual affective intensity for ADHD, Autism, and the three baseline terms. It reuses the frame-aware NRC-VAD collocate handoff built in the Sentiment notebook and computes Baes-style annual arousal indices from local target-window collocates.


## Setup

The diachronic axis is publication year (`lsc_year`). Target estimates are reported for the substantive core aggregate and for clinical-only, lived-only, and mixed frame strata. Baseline terms remain unframed comparator series.


In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#263238",
        "axes.labelcolor": "#263238",
        "axes.titlecolor": "#263238",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#D7DEE2",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "legend.frameon": False,
        "xtick.color": "#263238",
        "ytick.color": "#263238",
    }
)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
VAD_MATCH_PATH = PROJECT_ROOT / "data/interim/lsc/vad/lsc_vad_collocate_matches.parquet"
VAD_COVERAGE_PATH = PROJECT_ROOT / "data/interim/lsc/vad/lsc_vad_context_coverage.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/lsc/intensity"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/intensity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ANNUAL_AROUSAL_PATH = OUTPUT_DIR / "lsc_intensity_annual_arousal.csv"
COVERAGE_PATH = OUTPUT_DIR / "lsc_intensity_coverage.csv"
TOP_COLLOCATES_PATH = OUTPUT_DIR / "lsc_intensity_top_collocates.csv"
AUDIT_FLAGS_PATH = OUTPUT_DIR / "lsc_intensity_audit_flags.csv"
TREND_SUMMARY_PATH = OUTPUT_DIR / "lsc_intensity_trend_models.csv"
TRAJECTORY_PLOT_PATH = FIGURE_DIR / "lsc_intensity_arousal_trajectories.png"

EXPECTED_YEARS = list(range(2014, 2027))
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_UNITS = TARGET_UNITS + BASELINE_UNITS
CORE_TARGET_FRAMES = ["clinical_only", "lived_only", "mixed"]
TARGET_FRAME_STRATA = ["substantive_core_overall", *CORE_TARGET_FRAMES]
BASELINE_FRAME_STRATUM = "unframed_baseline"
BOOTSTRAP_REPETITIONS = 500
RANDOM_SEED = 123
MIN_FRAME_CONTEXTS_FOR_INTERPRETATION = 100
MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION = 50
DW_AUTOCORRELATION_LOW = 1.25
DW_AUTOCORRELATION_HIGH = 2.75

UNIT_COLOURS = {
    "ADHD": "#2F6F9F",
    "Autism": "#B66A4A",
    "frustration": "#4F8F78",
    "loneliness": "#7FA68A",
    "sadness": "#9AA6A1",
}
UNIT_MARKERS = {
    "ADHD": "o",
    "Autism": "s",
    "frustration": "^",
    "loneliness": "D",
    "sadness": "v",
}
FRAME_LABELS = {
    "substantive_core_overall": "Overall",
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
    "mixed": "Mixed clinical/lived framing",
    "unframed_baseline": "Comparator term",
}
FRAME_COLORS = {
    "substantive_core_overall": "#263238",
    "clinical_only": "#4F8DB3",
    "lived_only": "#C98263",
    "mixed": "#79A889",
    "substantive_other": "#A998C9",
    "non_substantive_or_insufficient": "#B8C0C5",
    "unframed_baseline": "#7B8785",
}
CONDITION_FRAME_COLORS = {
    "ADHD": {
        "substantive_core_overall": "#2F6F9F",
        "clinical_only": "#75A9C8",
        "lived_only": "#AECFE0",
        "mixed": "#D4E4EC",
    },
    "Autism": {
        "substantive_core_overall": "#B66A4A",
        "clinical_only": "#CE8D70",
        "lived_only": "#E1B49D",
        "mixed": "#F2D8CF",
    },
}
FRAME_MARKERS = {
    "substantive_core_overall": "o",
    "clinical_only": "s",
    "lived_only": "^",
    "mixed": "D",
    "unframed_baseline": "o",
}
LSC_FIGURE_DPI = 300


## Load VAD Handoff

The handoff contains one row per matched collocate occurrence and one coverage row per analysis context. It is produced by the Sentiment notebook, so the tokenisation, lemmatisation, focal-term exclusion, MWE matching, and frame-stratum contract remain identical for Sentiment and Intensity.


In [2]:
if not VAD_MATCH_PATH.exists():
    raise FileNotFoundError(f"Missing VAD match handoff: {VAD_MATCH_PATH}")
if not VAD_COVERAGE_PATH.exists():
    raise FileNotFoundError(f"Missing VAD coverage handoff: {VAD_COVERAGE_PATH}")

vad_matches = pd.read_parquet(VAD_MATCH_PATH)
context_coverage = pd.read_parquet(VAD_COVERAGE_PATH)

required_match_columns = {
    "context_row_id",
    "doc_id",
    "lsc_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "frame_stratum",
    "registered_domain",
    "collocate",
    "valence",
    "arousal",
    "dominance",
}
required_coverage_columns = {
    "context_row_id",
    "doc_id",
    "lsc_year",
    "analysis_unit",
    "frame_stratum",
    "candidate_collocate_tokens",
    "matched_vad_units",
    "matched_token_positions",
    "has_vad_match",
}
missing_match_columns = sorted(required_match_columns - set(vad_matches.columns))
missing_coverage_columns = sorted(required_coverage_columns - set(context_coverage.columns))
if missing_match_columns:
    raise RuntimeError(f"VAD match handoff is missing columns: {missing_match_columns}")
if missing_coverage_columns:
    raise RuntimeError(f"VAD coverage handoff is missing columns: {missing_coverage_columns}")

observed_units = sorted(context_coverage["analysis_unit"].dropna().unique())
observed_years = sorted(context_coverage["lsc_year"].dropna().astype(int).unique())
observed_frame_strata = sorted(context_coverage["frame_stratum"].dropna().unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
missing_target_strata = sorted(set(TARGET_FRAME_STRATA) - set(observed_frame_strata))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units in VAD coverage handoff: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years in VAD coverage handoff: {missing_years}")
if missing_target_strata:
    raise RuntimeError(f"Missing expected target frame strata in VAD coverage handoff: {missing_target_strata}")

handoff_summary = pd.DataFrame(
    {
        "metric": ["matched_vad_collocate_rows", "context_rows", "documents", "analysis_units", "frame_strata", "years"],
        "value": [
            len(vad_matches),
            len(context_coverage),
            context_coverage["doc_id"].nunique(),
            ", ".join(observed_units),
            ", ".join(observed_frame_strata),
            f"{min(observed_years)}-{max(observed_years)}",
        ],
    }
)
handoff_summary


,metric,value
0,matched_vad_collocate_rows,1972151
1,context_rows,311030
2,documents,192046
3,analysis_units,"ADHD, Autism, frustration, loneliness, sadness"
4,frame_strata,"clinical_only, lived_only, mixed, substantive_core_overall, unframed_baseline"
5,years,2014-2026


## Annual Arousal Index

Annual arousal is the weighted mean of all matched NRC-VAD collocate occurrences for each analysis unit, publication year, and frame stratum. Coverage and small-cell flags are carried alongside the index.


In [3]:
GROUP_COLUMNS = ["lsc_year", "analysis_unit", "frame_stratum"]

coverage = (
    context_coverage.groupby(GROUP_COLUMNS, as_index=False)
    .agg(
        context_rows=("context_row_id", "size"),
        documents=("doc_id", "nunique"),
        candidate_collocate_tokens=("candidate_collocate_tokens", "sum"),
        matched_vad_units_coverage=("matched_vad_units", "sum"),
        matched_token_positions=("matched_token_positions", "sum"),
        contexts_with_vad_match=("has_vad_match", "sum"),
    )
)
coverage["matched_token_coverage"] = coverage["matched_token_positions"] / coverage["candidate_collocate_tokens"].replace(0, np.nan)
coverage["context_match_coverage"] = coverage["contexts_with_vad_match"] / coverage["context_rows"].replace(0, np.nan)
coverage["small_cell_flag"] = coverage["frame_stratum"].isin(CORE_TARGET_FRAMES) & (
    coverage["context_rows"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | coverage["documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)

annual_arousal = (
    vad_matches.groupby([*GROUP_COLUMNS, "term_role", "target_group"], as_index=False)
    .agg(
        arousal_mean=("arousal", "mean"),
        arousal_sd=("arousal", "std"),
        valence_mean_for_reference=("valence", "mean"),
        dominance_mean_for_reference=("dominance", "mean"),
        matched_vad_units=("arousal", "size"),
        unique_collocates=("collocate", "nunique"),
        documents_with_matches=("doc_id", "nunique"),
    )
)
annual_arousal = annual_arousal.merge(coverage, on=GROUP_COLUMNS, how="left")
annual_arousal = annual_arousal.sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).reset_index(drop=True)
annual_arousal.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,arousal_mean,arousal_sd,valence_mean_for_reference,dominance_mean_for_reference,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,small_cell_flag
0,2014,ADHD,clinical_only,target,ADHD,-0.027904,0.279504,0.060713,0.015758,7410,1406,807,1286,808,9069,7410,7935,1273,0.874959,0.989891,False
1,2015,ADHD,clinical_only,target,ADHD,-0.032178,0.276076,0.066554,0.015869,6765,1379,771,1203,774,8360,6765,7231,1191,0.864952,0.990025,False
2,2016,ADHD,clinical_only,target,ADHD,-0.021915,0.278681,0.044939,0.008918,6532,1357,789,1149,790,8030,6532,7025,1141,0.874844,0.993037,False
3,2017,ADHD,clinical_only,target,ADHD,-0.018393,0.279832,0.048403,0.014013,6178,1346,748,1099,751,7593,6178,6602,1087,0.869485,0.989081,False
4,2018,ADHD,clinical_only,target,ADHD,-0.020456,0.281968,0.048075,0.003309,6869,1425,785,1182,787,8408,6869,7339,1176,0.872859,0.994924,False
5,2019,ADHD,clinical_only,target,ADHD,-0.013096,0.285849,0.029206,0.000834,5789,1218,672,1000,674,7120,5789,6196,995,0.870225,0.995000,False
6,2020,ADHD,clinical_only,target,ADHD,-0.022576,0.276691,0.044193,0.007805,5421,1179,665,959,668,6722,5421,5833,947,0.867748,0.987487,False
7,2021,ADHD,clinical_only,target,ADHD,-0.015004,0.296847,0.037715,0.001729,5281,1252,635,928,636,6430,5281,5639,922,0.876983,0.993534,False
8,2022,ADHD,clinical_only,target,ADHD,-0.027568,0.280115,0.057969,0.021404,5392,1212,633,930,634,6550,5392,5797,923,0.885038,0.992473,False
9,2023,ADHD,clinical_only,target,ADHD,-0.011918,0.283296,0.053057,0.023341,4241,1053,474,750,477,5142,4241,4514,736,0.877869,0.981333,False


## Bootstrap Confidence Intervals

The bootstrap resamples documents within each unit-year-frame stratum. This matches the Sentiment uncertainty contract and avoids treating collocates from the same document as independent observations.


In [4]:
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_records: list[dict[str, object]] = []

doc_scores = (
    vad_matches.groupby([*GROUP_COLUMNS, "doc_id"], as_index=False)
    .agg(arousal_sum=("arousal", "sum"), matched_vad_units=("arousal", "size"))
)

for group_values, frame in doc_scores.groupby(GROUP_COLUMNS, sort=True):
    arousal_sums = frame["arousal_sum"].to_numpy(dtype=float)
    unit_counts = frame["matched_vad_units"].to_numpy(dtype=float)
    n_docs = len(frame)
    estimates = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    for _ in range(BOOTSTRAP_REPETITIONS):
        sample_index = rng.integers(0, n_docs, size=n_docs)
        denominator = unit_counts[sample_index].sum()
        estimates[_] = arousal_sums[sample_index].sum() / denominator if denominator else np.nan
    bootstrap_records.append(
        {
            "lsc_year": int(group_values[0]),
            "analysis_unit": group_values[1],
            "frame_stratum": group_values[2],
            "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
            "bootstrap_unit": "doc_id",
            "arousal_bootstrap_mean": float(np.nanmean(estimates)),
            "arousal_ci_low": float(np.nanquantile(estimates, 0.025)),
            "arousal_ci_high": float(np.nanquantile(estimates, 0.975)),
        }
    )

bootstrap_arousal = pd.DataFrame(bootstrap_records)
annual_arousal = annual_arousal.merge(bootstrap_arousal, on=GROUP_COLUMNS, how="left")
annual_arousal.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,arousal_mean,arousal_sd,valence_mean_for_reference,dominance_mean_for_reference,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,small_cell_flag,bootstrap_repetitions,bootstrap_unit,arousal_bootstrap_mean,arousal_ci_low,arousal_ci_high
0,2014,ADHD,clinical_only,target,ADHD,-0.027904,0.279504,0.060713,0.015758,7410,1406,807,1286,808,9069,7410,7935,1273,0.874959,0.989891,False,500,doc_id,-0.027717,-0.034123,-0.020622
1,2015,ADHD,clinical_only,target,ADHD,-0.032178,0.276076,0.066554,0.015869,6765,1379,771,1203,774,8360,6765,7231,1191,0.864952,0.990025,False,500,doc_id,-0.031953,-0.039842,-0.024182
2,2016,ADHD,clinical_only,target,ADHD,-0.021915,0.278681,0.044939,0.008918,6532,1357,789,1149,790,8030,6532,7025,1141,0.874844,0.993037,False,500,doc_id,-0.022050,-0.028870,-0.014596
3,2017,ADHD,clinical_only,target,ADHD,-0.018393,0.279832,0.048403,0.014013,6178,1346,748,1099,751,7593,6178,6602,1087,0.869485,0.989081,False,500,doc_id,-0.018499,-0.025613,-0.011174
4,2018,ADHD,clinical_only,target,ADHD,-0.020456,0.281968,0.048075,0.003309,6869,1425,785,1182,787,8408,6869,7339,1176,0.872859,0.994924,False,500,doc_id,-0.020351,-0.027925,-0.012633
5,2019,ADHD,clinical_only,target,ADHD,-0.013096,0.285849,0.029206,0.000834,5789,1218,672,1000,674,7120,5789,6196,995,0.870225,0.995000,False,500,doc_id,-0.012915,-0.021260,-0.004591
6,2020,ADHD,clinical_only,target,ADHD,-0.022576,0.276691,0.044193,0.007805,5421,1179,665,959,668,6722,5421,5833,947,0.867748,0.987487,False,500,doc_id,-0.022767,-0.031536,-0.013694
7,2021,ADHD,clinical_only,target,ADHD,-0.015004,0.296847,0.037715,0.001729,5281,1252,635,928,636,6430,5281,5639,922,0.876983,0.993534,False,500,doc_id,-0.015037,-0.024421,-0.005970
8,2022,ADHD,clinical_only,target,ADHD,-0.027568,0.280115,0.057969,0.021404,5392,1212,633,930,634,6550,5392,5797,923,0.885038,0.992473,False,500,doc_id,-0.027379,-0.035534,-0.018471
9,2023,ADHD,clinical_only,target,ADHD,-0.011918,0.283296,0.053057,0.023341,4241,1053,474,750,477,5142,4241,4514,736,0.877869,0.981333,False,500,doc_id,-0.011766,-0.021248,-0.002405


## Trend Models

The main descriptive trend is OLS arousal-on-centred-year for each reported series. Residual autocorrelation is flagged with a Durbin-Watson diagnostic; when flagged, an AR(1)-transformed sensitivity slope is reported in the trend table.


In [5]:
def fit_trend(frame: pd.DataFrame, value_column: str) -> dict[str, object]:
    data = frame[["lsc_year", value_column]].dropna().sort_values("lsc_year")
    if len(data) < 3 or data[value_column].nunique() < 2:
        return {
            "n_years": len(data),
            "year_center": np.nan,
            "linear_intercept": np.nan,
            "linear_slope_per_year": np.nan,
            "linear_slope_se": np.nan,
            "linear_p_value": np.nan,
            "linear_r_squared": np.nan,
            "linear_adj_r_squared": np.nan,
            "standardized_beta_year": np.nan,
            "durbin_watson": np.nan,
            "lag1_residual_autocorrelation": np.nan,
            "autocorrelation_flag": False,
            "ar1_sensitivity_slope_per_year": np.nan,
            "ar1_sensitivity_p_value": np.nan,
            "quadratic_adj_r_squared": np.nan,
            "quadratic_delta_adj_r_squared": np.nan,
        }
    years = data["lsc_year"].to_numpy(dtype=float)
    values = data[value_column].to_numpy(dtype=float)
    year_center = float(years.mean())
    x = years - year_center
    result = stats.linregress(x, values)
    fitted = result.intercept + result.slope * x
    residuals = values - fitted
    sse = float(np.sum(residuals**2))
    sst = float(np.sum((values - values.mean()) ** 2))
    r_squared = 1 - sse / sst if sst else np.nan
    n = len(values)
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2) if n > 2 and not np.isnan(r_squared) else np.nan
    std_beta = result.slope * np.std(x, ddof=1) / np.std(values, ddof=1) if np.std(values, ddof=1) else np.nan
    dw_denominator = float(np.sum(residuals**2))
    durbin_watson = float(np.sum(np.diff(residuals) ** 2) / dw_denominator) if dw_denominator else np.nan
    lag1 = float(np.corrcoef(residuals[:-1], residuals[1:])[0, 1]) if n >= 4 and np.std(residuals) else np.nan
    autocorrelation_flag = bool(
        not np.isnan(durbin_watson)
        and (durbin_watson < DW_AUTOCORRELATION_LOW or durbin_watson > DW_AUTOCORRELATION_HIGH)
    )
    ar1_slope = np.nan
    ar1_p_value = np.nan
    if autocorrelation_flag and n >= 5 and not np.isnan(lag1) and abs(lag1) < 0.98:
        y_star = values[1:] - lag1 * values[:-1]
        x_star = x[1:] - lag1 * x[:-1]
        ar1_result = stats.linregress(x_star, y_star)
        ar1_slope = float(ar1_result.slope)
        ar1_p_value = float(ar1_result.pvalue)
    quadratic_adj_r_squared = np.nan
    quadratic_delta = np.nan
    if n >= 6:
        q_coefficients = np.polyfit(x, values, deg=2)
        q_fitted = np.polyval(q_coefficients, x)
        q_sse = float(np.sum((values - q_fitted) ** 2))
        q_r_squared = 1 - q_sse / sst if sst else np.nan
        quadratic_adj_r_squared = 1 - (1 - q_r_squared) * (n - 1) / (n - 3) if n > 3 and not np.isnan(q_r_squared) else np.nan
        quadratic_delta = quadratic_adj_r_squared - adj_r_squared if not np.isnan(quadratic_adj_r_squared) else np.nan
    return {
        "n_years": n,
        "year_center": year_center,
        "linear_intercept": float(result.intercept),
        "linear_slope_per_year": float(result.slope),
        "linear_slope_se": float(result.stderr) if result.stderr is not None else np.nan,
        "linear_p_value": float(result.pvalue),
        "linear_r_squared": float(r_squared),
        "linear_adj_r_squared": float(adj_r_squared),
        "standardized_beta_year": float(std_beta),
        "durbin_watson": durbin_watson,
        "lag1_residual_autocorrelation": lag1,
        "autocorrelation_flag": autocorrelation_flag,
        "ar1_sensitivity_slope_per_year": ar1_slope,
        "ar1_sensitivity_p_value": ar1_p_value,
        "quadratic_adj_r_squared": quadratic_adj_r_squared,
        "quadratic_delta_adj_r_squared": quadratic_delta,
    }


trend_rows = []
for group_values, frame in annual_arousal.groupby(["analysis_unit", "frame_stratum", "term_role", "target_group"], sort=True):
    analysis_unit, frame_stratum, term_role, target_group = group_values
    trend_rows.append(
        {
            "analysis_unit": analysis_unit,
            "frame_stratum": frame_stratum,
            "term_role": term_role,
            "target_group": target_group,
            "index_name": "arousal_mean",
            **fit_trend(frame, "arousal_mean"),
        }
    )
trend_summary = pd.DataFrame(trend_rows)
trend_summary.to_csv(TREND_SUMMARY_PATH, index=False)
trend_summary.head(12)


,analysis_unit,frame_stratum,term_role,target_group,index_name,n_years,year_center,linear_intercept,linear_slope_per_year,linear_slope_se,linear_p_value,linear_r_squared,linear_adj_r_squared,standardized_beta_year,durbin_watson,lag1_residual_autocorrelation,autocorrelation_flag,ar1_sensitivity_slope_per_year,ar1_sensitivity_p_value,quadratic_adj_r_squared,quadratic_delta_adj_r_squared
0,ADHD,clinical_only,target,ADHD,arousal_mean,13,2020.0,-0.019026,0.001311,0.000414,0.008922,0.477379,0.429868,0.690926,2.718894,-0.389642,False,NaN,NaN,0.385572,-0.044296
1,ADHD,lived_only,target,ADHD,arousal_mean,13,2020.0,-0.039346,0.001364,0.000677,0.069087,0.269466,0.203053,0.519101,1.639166,0.144783,False,NaN,NaN,0.125387,-0.077666
2,ADHD,mixed,target,ADHD,arousal_mean,13,2020.0,-0.032164,0.000687,0.000810,0.414635,0.061330,-0.024004,0.247648,1.806004,0.075630,False,NaN,NaN,0.084156,0.108160
3,ADHD,substantive_core_overall,target,ADHD,arousal_mean,13,2020.0,-0.025004,0.001013,0.000374,0.020419,0.399728,0.345158,0.632241,1.881084,0.049988,False,NaN,NaN,0.297337,-0.047822
4,Autism,clinical_only,target,Autism,arousal_mean,13,2020.0,-0.014219,0.001099,0.000295,0.003305,0.558878,0.518776,0.747581,2.179659,-0.423790,False,NaN,NaN,0.533204,0.014428
5,Autism,lived_only,target,Autism,arousal_mean,13,2020.0,-0.044556,0.001679,0.001162,0.176250,0.159595,0.083195,0.399494,2.260692,-0.153070,False,NaN,NaN,0.134866,0.051671
6,Autism,mixed,target,Autism,arousal_mean,13,2020.0,-0.039163,0.001074,0.000556,0.079550,0.253309,0.185428,0.503298,1.924830,-0.108680,False,NaN,NaN,0.145283,-0.040145
7,Autism,substantive_core_overall,target,Autism,arousal_mean,13,2020.0,-0.028354,0.001137,0.000559,0.066880,0.273158,0.207081,0.522645,2.345848,-0.236306,False,NaN,NaN,0.202412,-0.004669
8,frustration,unframed_baseline,baseline,baseline,arousal_mean,13,2020.0,-0.024250,-0.000261,0.000212,0.244023,0.121067,0.041164,-0.347946,0.608792,0.617079,True,-0.000999,0.049705,0.477426,0.436263
9,loneliness,unframed_baseline,baseline,baseline,arousal_mean,13,2020.0,-0.023863,0.000076,0.000358,0.836446,0.004046,-0.086495,0.063608,1.822414,0.080460,False,NaN,NaN,0.133562,0.220057


## Diagnostics And Save Tables

Diagnostics flag sparse frame-year cells, low VAD coverage, high collocate concentration, and trend-series residual autocorrelation. They are interpretive warnings rather than exclusion rules.


In [6]:
collocate_counts = (
    vad_matches.groupby(["lsc_year", "analysis_unit", "frame_stratum", "collocate"], as_index=False)
    .agg(
        occurrences=("collocate", "size"),
        arousal=("arousal", "mean"),
        valence=("valence", "mean"),
        documents=("doc_id", "nunique"),
    )
)
collocate_totals = collocate_counts.groupby(["lsc_year", "analysis_unit", "frame_stratum"])["occurrences"].transform("sum")
collocate_counts["occurrence_share"] = collocate_counts["occurrences"] / collocate_totals
collocate_counts["weighted_arousal_contribution"] = collocate_counts["occurrences"] * collocate_counts["arousal"]

collocate_counts["total_matches_for_unit_year_frame"] = collocate_totals
top_collocates = (
    collocate_counts.sort_values(["lsc_year", "analysis_unit", "frame_stratum", "occurrences"], ascending=[True, True, True, False])
    .groupby(["lsc_year", "analysis_unit", "frame_stratum"], as_index=False)
    .head(15)
    .reset_index(drop=True)
)

concentration = (
    collocate_counts.sort_values(["lsc_year", "analysis_unit", "frame_stratum", "occurrences"], ascending=[True, True, True, False])
    .groupby(["lsc_year", "analysis_unit", "frame_stratum"], as_index=False)
    .head(5)
    .groupby(["lsc_year", "analysis_unit", "frame_stratum"], as_index=False)
    .agg(top5_collocate_share=("occurrence_share", "sum"))
)

coverage_for_flags = annual_arousal.merge(concentration, on=GROUP_COLUMNS, how="left")
flag_rows = []
for row in coverage_for_flags.itertuples(index=False):
    flags = []
    if bool(row.small_cell_flag):
        flags.append("small_frame_year_cell")
    if row.context_match_coverage < 0.90:
        flags.append("low_context_vad_match_coverage_lt_0_90")
    if row.matched_token_coverage < 0.70:
        flags.append("low_token_vad_coverage_lt_0_70")
    if pd.notna(row.top5_collocate_share) and row.top5_collocate_share > 0.35:
        flags.append("top5_collocate_share_gt_0_35")
    if flags:
        flag_rows.append(
            {
                "lsc_year": row.lsc_year,
                "analysis_unit": row.analysis_unit,
                "frame_stratum": row.frame_stratum,
                "flags": ";".join(flags),
                "context_rows": row.context_rows,
                "documents": row.documents,
                "context_match_coverage": row.context_match_coverage,
                "matched_token_coverage": row.matched_token_coverage,
                "top5_collocate_share": row.top5_collocate_share,
            }
        )

for row in trend_summary.loc[trend_summary["autocorrelation_flag"]].itertuples(index=False):
    flag_rows.append(
        {
            "lsc_year": pd.NA,
            "analysis_unit": row.analysis_unit,
            "frame_stratum": row.frame_stratum,
            "flags": "trend_residual_autocorrelation",
            "context_rows": pd.NA,
            "documents": pd.NA,
            "context_match_coverage": pd.NA,
            "matched_token_coverage": pd.NA,
            "top5_collocate_share": pd.NA,
        }
    )

audit_flag_columns = [
    "lsc_year",
    "analysis_unit",
    "frame_stratum",
    "flags",
    "context_rows",
    "documents",
    "context_match_coverage",
    "matched_token_coverage",
    "top5_collocate_share",
]
audit_flags = pd.DataFrame(flag_rows, columns=audit_flag_columns)

annual_arousal.to_csv(ANNUAL_AROUSAL_PATH, index=False)
coverage_for_flags.to_csv(COVERAGE_PATH, index=False)
top_collocates.to_csv(TOP_COLLOCATES_PATH, index=False)
audit_flags.to_csv(AUDIT_FLAGS_PATH, index=False)

pd.DataFrame(
    {
        "output": ["annual_arousal", "coverage", "top_collocates", "trend_summary", "audit_flags"],
        "path": [ANNUAL_AROUSAL_PATH, COVERAGE_PATH, TOP_COLLOCATES_PATH, TREND_SUMMARY_PATH, AUDIT_FLAGS_PATH],
        "rows": [len(annual_arousal), len(coverage_for_flags), len(top_collocates), len(trend_summary), len(audit_flags)],
    }
)


,output,path,rows
0,annual_arousal,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_annual_arousal.csv,143
1,coverage,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_coverage.csv,143
2,top_collocates,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_top_collocates.csv,2145
3,trend_summary,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_trend_models.csv,11
4,audit_flags,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_audit_flags.csv,2


## Arousal Trajectories

The report-facing trajectory figure uses three equal-width panels: ADHD, Autism, and comparator terms. The ADHD and Autism panels foreground the substantive-core Overall trajectory and add clinical/disorder and lived-experience traces as lighter contextual lines.

Mixed-frame estimates, coverage diagnostics, small-cell warnings, collocate concentration, and trend diagnostics remain available in the saved CSV tables and audit flags. They are not saved as separate report figures in order to keep the figure folder aligned with the main dissertation story.


In [7]:
READER_FRAME_STRATA = ["clinical_only", "lived_only"]
READER_FRAME_LABELS = {
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
}


def save_lsc_figure(fig: plt.Figure, png_path: Path) -> Path:
    fig.tight_layout(pad=1.1, rect=[0, 0, 1, 0.93])
    fig.savefig(png_path, dpi=LSC_FIGURE_DPI, bbox_inches="tight", facecolor="white")
    pdf_path = png_path.with_suffix(".pdf")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    return pdf_path


def series_color(unit: str, frame_stratum: str) -> str:
    if unit in CONDITION_FRAME_COLORS and frame_stratum in CONDITION_FRAME_COLORS[unit]:
        return CONDITION_FRAME_COLORS[unit][frame_stratum]
    if unit in UNIT_COLOURS:
        return UNIT_COLOURS[unit]
    return FRAME_COLORS.get(frame_stratum, "#7B8785")


def trend_line_for(series: pd.DataFrame, trend: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    years = series["lsc_year"].to_numpy(dtype=float)
    fitted = trend["linear_intercept"] + trend["linear_slope_per_year"] * (years - trend["year_center"])
    return years, fitted


def trend_for(unit: str, frame_stratum: str) -> pd.Series | None:
    row = trend_summary.loc[
        trend_summary["analysis_unit"].eq(unit) & trend_summary["frame_stratum"].eq(frame_stratum)
    ]
    if row.empty or pd.isna(row.iloc[0]["linear_slope_per_year"]):
        return None
    return row.iloc[0]


def y_limits_from(frame: pd.DataFrame, value_column: str, ci_low: str | None = None, ci_high: str | None = None) -> tuple[float, float]:
    values = [frame[value_column].to_numpy(dtype=float)]
    if ci_low and ci_low in frame:
        values.append(frame[ci_low].to_numpy(dtype=float))
    if ci_high and ci_high in frame:
        values.append(frame[ci_high].to_numpy(dtype=float))
    finite_values = [v[np.isfinite(v)] for v in values if len(v)]
    combined = np.concatenate(finite_values)
    low, high = float(combined.min()), float(combined.max())
    padding = max((high - low) * 0.10, 0.004)
    return low - padding, high + padding


def style_year_axis(ax: plt.Axes) -> None:
    ax.set_xticks(EXPECTED_YEARS[::2])
    ax.tick_params(axis="x", labelsize=8.5)


def plot_line_with_trend(
    ax: plt.Axes,
    frame: pd.DataFrame,
    unit: str,
    frame_stratum: str,
    value_column: str,
    color: str,
    marker: str,
    label: str | None = None,
    ci_low: str | None = None,
    ci_high: str | None = None,
    ribbon_alpha: float = 0.12,
    linewidth: float = 2.2,
    markersize: float = 4.8,
    alpha: float = 1.0,
    show_trend: bool = True,
) -> None:
    series = frame.loc[frame["analysis_unit"].eq(unit) & frame["frame_stratum"].eq(frame_stratum)].sort_values("lsc_year")
    if series.empty:
        return
    ax.plot(
        series["lsc_year"],
        series[value_column],
        marker=marker,
        markersize=markersize,
        linewidth=linewidth,
        label=label,
        color=color,
        alpha=alpha,
    )
    if ci_low and ci_high:
        ax.fill_between(
            series["lsc_year"].to_numpy(dtype=float),
            series[ci_low].to_numpy(dtype=float),
            series[ci_high].to_numpy(dtype=float),
            color=color,
            alpha=ribbon_alpha,
            linewidth=0,
        )
    trend = trend_for(unit, frame_stratum)
    if show_trend and trend is not None:
        years, fitted = trend_line_for(series, trend)
        ax.plot(years, fitted, color=color, linewidth=1.05, linestyle="--", alpha=min(alpha + 0.12, 0.92))


def plot_target_panel(ax: plt.Axes, unit: str) -> None:
    plot_line_with_trend(
        ax,
        annual_arousal,
        unit,
        "substantive_core_overall",
        "arousal_mean",
        series_color(unit, "substantive_core_overall"),
        FRAME_MARKERS["substantive_core_overall"],
        "Overall",
        "arousal_ci_low",
        "arousal_ci_high",
        ribbon_alpha=0.14,
        linewidth=2.8,
        markersize=4.8,
    )
    for frame_stratum in READER_FRAME_STRATA:
        plot_line_with_trend(
            ax,
            annual_arousal,
            unit,
            frame_stratum,
            "arousal_mean",
            series_color(unit, frame_stratum),
            FRAME_MARKERS[frame_stratum],
            READER_FRAME_LABELS[frame_stratum],
            "arousal_ci_low",
            "arousal_ci_high",
            ribbon_alpha=0.075,
            linewidth=1.55,
            markersize=3.7,
            alpha=0.82,
        )
    ax.set_title(unit, loc="left", fontsize=11, fontweight="bold")
    style_year_axis(ax)
    ax.legend(loc="best", fontsize=7.6)


def plot_baseline_panel(
    ax: plt.Axes,
    frame: pd.DataFrame,
    value_column: str,
    ci_low: str | None = None,
    ci_high: str | None = None,
    show_trend: bool = True,
) -> None:
    for unit in BASELINE_UNITS:
        plot_line_with_trend(
            ax,
            frame,
            unit,
            BASELINE_FRAME_STRATUM,
            value_column,
            UNIT_COLOURS[unit],
            UNIT_MARKERS[unit] if "UNIT_MARKERS" in globals() else LSC_UNIT_MARKERS[unit],
            LSC_UNIT_LABELS[unit] if "LSC_UNIT_LABELS" in globals() else unit,
            ci_low,
            ci_high,
            ribbon_alpha=0.08,
            linewidth=2.0,
            show_trend=show_trend,
        )
    ax.set_title("Comparator terms", loc="left", fontsize=11, fontweight="bold")
    ax.legend(loc="best", fontsize=7.6)
    style_year_axis(ax)


main_rows = pd.concat(
    [
        annual_arousal.loc[
            annual_arousal["analysis_unit"].isin(TARGET_UNITS)
            & annual_arousal["frame_stratum"].isin(["substantive_core_overall", *READER_FRAME_STRATA])
        ],
        annual_arousal.loc[annual_arousal["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    ],
    ignore_index=True,
)
main_ylim = y_limits_from(main_rows, "arousal_mean", "arousal_ci_low", "arousal_ci_high")

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.35), sharex=True, sharey=True)
fig.suptitle("Intensity: arousal near target terms", fontsize=14, fontweight="bold", x=0.02, ha="left")
for ax, unit in zip(axes[:2], TARGET_UNITS):
    plot_target_panel(ax, unit)
plot_baseline_panel(
    axes[2],
    annual_arousal.loc[annual_arousal["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    "arousal_mean",
    "arousal_ci_low",
    "arousal_ci_high",
)
for ax in axes:
    ax.set_ylim(*main_ylim)
    ax.set_xlabel("Publication year")
axes[0].set_ylabel("Mean arousal (-1 to 1)")
trajectory_pdf = save_lsc_figure(fig, TRAJECTORY_PLOT_PATH)
plt.close(fig)



## Coverage Overview

Coverage is high when most contexts produce at least one NRC-VAD match and when a large share of candidate collocate token positions are matched. These diagnostics should be checked before interpreting sharp frame-specific movements.


In [8]:
if "audit_flags" not in coverage_for_flags.columns:
    coverage_for_flags = coverage_for_flags.assign(audit_flags="")

coverage_summary = (
    coverage_for_flags.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(
        min_context_match_coverage=("context_match_coverage", "min"),
        min_matched_token_coverage=("matched_token_coverage", "min"),
        years_with_audit_flags=("audit_flags", lambda values: int(values.astype(str).ne("").sum())),
    )
    .sort_values(["analysis_unit", "frame_stratum"])
)

print("Coverage diagnostics are saved as CSV rather than a report figure:")
print(f"- {COVERAGE_PATH.relative_to(PROJECT_ROOT)}")
display(coverage_summary.round(3))


Coverage diagnostics are saved as CSV rather than a report figure:
- data/processed/lsc/intensity/lsc_intensity_coverage.csv


,analysis_unit,frame_stratum,min_context_match_coverage,min_matched_token_coverage,years_with_audit_flags
0,ADHD,clinical_only,0.981,0.856,0
1,ADHD,lived_only,0.981,0.843,0
2,ADHD,mixed,0.982,0.839,0
3,ADHD,substantive_core_overall,0.983,0.859,0
4,Autism,clinical_only,0.998,0.847,0
5,Autism,lived_only,0.997,0.831,0
6,Autism,mixed,0.997,0.835,0
7,Autism,substantive_core_overall,0.998,0.843,0
8,frustration,unframed_baseline,0.999,0.835,0
9,loneliness,unframed_baseline,0.999,0.845,0


## Handoff Summary

The handoff summary reports the mean, sample standard deviation, and range of the annual arousal estimates alongside coverage, warning counts, and trend slopes. The descriptive mean and standard deviation summarise annual trajectory values rather than collocate- or context-level observations.


In [9]:
summary = (
    annual_arousal.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(
        years=("lsc_year", "nunique"),
        arousal_annual_mean=("arousal_mean", "mean"),
        arousal_annual_sd=("arousal_mean", "std"),
        arousal_min=("arousal_mean", "min"),
        arousal_max=("arousal_mean", "max"),
        matched_units=("matched_vad_units", "sum"),
        documents_with_matches=("documents_with_matches", "sum"),
        min_context_match_coverage=("context_match_coverage", "min"),
        min_matched_token_coverage=("matched_token_coverage", "min"),
        small_cell_years=("small_cell_flag", "sum"),
    )
)
warning_counts = (
    audit_flags.groupby(["analysis_unit", "frame_stratum"]).size().rename("warnings").reset_index()
    if not audit_flags.empty
    else pd.DataFrame({"analysis_unit": [], "frame_stratum": [], "warnings": []})
)
summary = summary.merge(warning_counts, on=["analysis_unit", "frame_stratum"], how="left").fillna({"warnings": 0})
summary = summary.merge(
    trend_summary[["analysis_unit", "frame_stratum", "linear_slope_per_year", "linear_p_value", "autocorrelation_flag"]],
    on=["analysis_unit", "frame_stratum"],
    how="left",
)
summary["warnings"] = summary["warnings"].astype(int)
summary["small_cell_years"] = summary["small_cell_years"].astype(int)
expected_rows = len(EXPECTED_YEARS) * (len(BASELINE_UNITS) + len(TARGET_UNITS) * len(TARGET_FRAME_STRATA))
if len(annual_arousal) != expected_rows:
    raise RuntimeError(f"Expected {expected_rows} annual rows, found {len(annual_arousal)}.")
summary


,analysis_unit,frame_stratum,years,arousal_annual_mean,arousal_annual_sd,arousal_min,arousal_max,matched_units,documents_with_matches,min_context_match_coverage,min_matched_token_coverage,small_cell_years,warnings,linear_slope_per_year,linear_p_value,autocorrelation_flag
0,ADHD,clinical_only,13,-0.019026,0.007392,-0.032178,-0.005700,68768,7982,0.981333,0.856495,0,0,0.001311,0.008922,False
1,ADHD,lived_only,13,-0.039346,0.010236,-0.055548,-0.022365,21835,2728,0.981481,0.842701,0,0,0.001364,0.069087,False
2,ADHD,mixed,13,-0.032164,0.010805,-0.051705,-0.012098,12258,1627,0.981818,0.839124,1,1,0.000687,0.414635,False
3,ADHD,substantive_core_overall,13,-0.025004,0.006239,-0.037325,-0.014466,102861,11420,0.983407,0.858857,0,0,0.001013,0.020419,False
4,Autism,clinical_only,13,-0.014219,0.005728,-0.023821,-0.001596,128618,12955,0.998168,0.846868,0,0,0.001099,0.003305,False
5,Autism,lived_only,13,-0.044556,0.016367,-0.053791,0.009366,80459,9066,0.997076,0.830529,0,0,0.001679,0.176250,False
6,Autism,mixed,13,-0.039163,0.008310,-0.050921,-0.020133,40768,4734,0.997312,0.835443,0,0,0.001074,0.079550,False
7,Autism,substantive_core_overall,13,-0.028354,0.008475,-0.033357,-0.001226,249845,24138,0.998140,0.842999,0,0,0.001137,0.066880,False
8,frustration,unframed_baseline,13,-0.024250,0.002920,-0.031939,-0.020765,686724,93653,0.998947,0.835410,0,1,-0.000261,0.244023,True
9,loneliness,unframed_baseline,13,-0.023863,0.004637,-0.029119,-0.011829,243581,30431,0.999417,0.845217,0,0,0.000076,0.836446,False
